`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print(OPENAI_API_KEY[:2])

UPSTAGE_API_KEY = os.getenv("UPSTAGE_API_KEY")
print(UPSTAGE_API_KEY[30:])

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
print(TAVILY_API_KEY[:4])

`(2) 기본 라이브러리`

In [12]:
import warnings
warnings.filterwarnings("ignore")

from langchain_community.vectorstores import FAISS
from langchain_core.messages import SystemMessage
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.tools import tool
from langchain_community.tools import TavilySearchResults

from langchain_openai import ChatOpenAI
from langchain_upstage import UpstageEmbeddings
from langchain_upstage import ChatUpstage

# LangGraph MessagesState라는 미리 만들어진 상태를 사용
from langgraph.graph import MessagesState
from langchain_core.documents import Document
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import create_react_agent

from textwrap import dedent
from typing import List, Literal, Tuple, TypedDict
from pydantic import BaseModel, Field

import gradio as gr

from pprint import pprint

import uuid

#from IPython.display import Image, display

###  2-1. Tool 정의

- 메뉴 검색을 위한 벡터저장소를 초기화 (기존 저장소를 로드)

In [6]:

embeddings_model = UpstageEmbeddings(model="solar-embedding-1-large")

# menu db 벡터 저장소 로드
menu_db = FAISS.load_local(
    "../db/menu_db", 
    embeddings_model, 
    allow_dangerous_deserialization=True
)

# wine db 벡터 저장소 로드
wine_db = FAISS.load_local(
    "../db/wine_db", 
    embeddings_model, 
    allow_dangerous_deserialization=True
)

@tool
def search_menu(query: str) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant menu information from the encrypted database.
    Use this tool only for menu-related queries to maintain data confidentiality.
    """
    docs = menu_db.similarity_search(query, k=6)
    if len(docs) > 0:
        return docs
    
    return [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]

@tool
def search_wine(query: str) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant wine information from the encrypted database.
    Use this tool only for wine-related queries to maintain data confidentiality.
    """
    docs = wine_db.similarity_search(query, k=6)
    if len(docs) > 0:
        return docs
    
    return [Document(page_content="관련 와인 정보를 찾을 수 없습니다.")]

# 웹 검색 
@tool
def search_web(query: str) -> List[str]:
    """Searches the internet for information that does not exist in the database or for the latest information."""

    tavily_search = TavilySearchResults(max_results=2)
    docs = tavily_search.invoke(query)

    formatted_docs = []
    for doc in docs:
        formatted_docs.append(
            Document(
                page_content= f'<Document href="{doc["url"]}"/>\n{doc["content"]}\n</Document>',
                metadata={"source": "web search", "url": doc["url"]}
                )
        )

    if len(formatted_docs) > 0:
        return formatted_docs
    
    return [Document(page_content="관련 정보를 찾을 수 없습니다.")]


# 도구 목록을 정의 
tools = [search_menu, search_wine, search_web]

### 2-2. LLM 모델
* bind_tools() 함수로 model 과 tool 연결

In [7]:
#from langchain_openai import ChatOpenAI

# 기본 LLM
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, streaming=True)

from langchain_upstage import ChatUpstage
llm = ChatUpstage(
        model="solar-pro",
        base_url="https://api.upstage.ai/v1",
        temperature=0.5
    )
print(llm.model_name)

# LLM에 도구 바인딩하여 추가 
llm_with_tools = llm.bind_tools(tools)

solar-pro


In [8]:
# 메뉴 검색에 관련된 질문을 하는 경우 -> 메뉴 검색 도구를 호출  
query = "대표 메뉴는 무엇인가요?"
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)
print("-" * 100)

pprint(ai_msg.content)
print("-" * 100)

pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='[The `search_menu` function is essential to retrieve the restaurant\'s authorized menu information, specifically the "대표 메뉴" (representative menu) items from the encrypted database. No other functions are necessary as the question is strictly about menu items.]', additional_kwargs={'tool_calls': [{'id': 'chatcmpl-tool-7cf8c37f13e547cd853b1cbd4c2d1f2d', 'function': {'arguments': '{"query": "\\ub300\\ud45c \\uba54\\ub274"}', 'name': 'search_menu'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 663, 'total_tokens': 729, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'solar-pro2-250909', 'system_fingerprint': None, 'id': '83a7d4c4-5cec-4043-9fa9-74da2013bd9a', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, 

In [9]:
# 도구들의 목적과 관련 없는 질문을 하는 경우 -> 도구 호출 없이 그대로 답변을 생성 
query = "안녕하세요?"
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)
print("-" * 100)

pprint(ai_msg.content)
print("-" * 100)

pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='안녕하세요! 오늘 무엇을 도와드릴까요? 메뉴, 와인, 또는 기타 정보가 필요하신가요? 필요한 것이 있으면 언제든지 말씀해주세요! 😊\n\n(기능 호출 없이 일반적인 인사와 도움 제안을 드렸습니다. 추가 질문이나 요청이 있을 경우 필요한 기능을 최소한으로 호출하겠습니다.)', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 660, 'total_tokens': 712, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'solar-pro2-250909', 'system_fingerprint': None, 'id': '43335790-0b58-4285-b2ff-44e641e238e6', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--5c5af66e-d5dc-49ef-98c6-85a003182892-0', usage_metadata={'input_tokens': 660, 'output_tokens': 52, 'total_tokens': 712, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})
--------------------------------------------------------------------

In [11]:
# 웹 검색 목적과 관련된 질문을 하는 경우 -> 웹 검색 도구 호출 
query = "2024년 상반기 엔비디아 시가총액은 어떻게 변동 되었나요?"
ai_msg = llm_with_tools.invoke(query)

pprint(ai_msg)
print("-" * 100)

pprint(ai_msg.content)
print("-" * 100)

pprint(ai_msg.tool_calls)
print("-" * 100)

AIMessage(content='[엔비디아의 2024년 상반기 시가총액 변동은 최신 금융 데이터가 필요하며, 이는 공개 데이터베이스나 웹 검색을 통해 확인해야 합니다. `search_web` 함수를 사용해 신뢰할 수 있는 금융/경제 사이트에서 정보를 검색하는 것이 필수적입니다.]  \n\n(참고: 실제 답변은 함수 호출 후 반환된 웹 검색 결과에 따라 달라집니다.)', additional_kwargs={'tool_calls': [{'id': 'chatcmpl-tool-ceb35d95b3a543d5bbd93849729f39c8', 'function': {'arguments': '{"query": "2024\\ub144 \\uc0c1\\ubc18\\uae30 \\uc5d4\\ube44\\ub514\\uc544 \\uc2dc\\uac00\\ucd1d\\uc561 \\ubcc0\\ub3d9"}', 'name': 'search_web'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 92, 'prompt_tokens': 675, 'total_tokens': 767, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'solar-pro2-250909', 'system_fingerprint': None, 'id': '08b01580-36cc-46b3-9b78-13745c0d400e', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}

## 3. Adaptive RAG


### 3-1. 그래프 구현

`(1) 상태 정의`

In [13]:
# 상태 Schema 정의 
class AdaptiveRagState(TypedDict):
    question: str
    documents: List[Document]
    generation: str

`(2) 질문 분석 -> 라우팅`
- 사용자의 질문을 분석하여 적절한 검색 방법을 선택 
- 레스토랑 메뉴 검색 or 레스토랑 와인 검색  or 일반 웹 검색 or 단순 답변

In [14]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# 라우팅 결정을 위한 데이터 모델
class ToolSelector(BaseModel):
    """Routes the user question to the most appropriate tool."""
    tool: Literal["search_menu", "search_web", "search_wine"] = Field(
        description="Select one of the tools: search_menu, search_wine or search_web based on the user's question.",
    )

# 구조화된 출력을 위한 LLM 설정
structured_llm = llm.with_structured_output(ToolSelector)

# 라우팅을 위한 프롬프트 템플릿
system = dedent("""You are an AI assistant specializing in routing user questions to the appropriate tool.
Use the following guidelines:
- For questions about the restaurant's menu, use the search_menu tool.
- For wine recommendations or pairing information, use the search_wine tool.
- For any other information or the most up-to-date data, use the search_web tool.
Always choose the most appropriate tool based on the user's question.""")

route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

# 질문 라우터 정의
question_router = route_prompt | structured_llm

# 테스트 실행
print(question_router.invoke({"question": "채식주의자를 위한 메뉴가 있나요?"}))
print(question_router.invoke({"question": "스테이크 메뉴와 어울리는 와인을 추천해주세요."}))
print(question_router.invoke({"question": "2022년 월드컵 우승 국가는 어디인가요?"}))

tool='search_menu'
tool='search_wine'
tool='search_web'


In [15]:
# 질문 라우팅 노드 
def route_question_adaptive(state: AdaptiveRagState) -> Literal["search_menu", "search_wine", "search_web", "llm_fallback"]:
    question = state["question"]
    try:
        result = question_router.invoke({"question": question})
        datasource = result.tool
        
        if datasource == "search_menu":
            return "search_menu"
        elif datasource == "search_wine":
            return "search_wine"        
        elif datasource == "search_web":
            return "search_web"
        else:
            return "llm_fallback"
    
    except Exception as e:
        print(f"Error in routing: {str(e)}")
        return "llm_fallback"

`(3) 검색 노드`

In [16]:
def search_menu_adaptive(state: AdaptiveRagState):
    """
    Node for searching information in the restaurant menu
    """
    question = state["question"]
    docs = search_menu.invoke(question)

    if len(docs) > 0:
        return {"documents": docs}
    else:
        return {"documents": [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]}


def search_wine_adaptive(state: AdaptiveRagState):
    """
    Node for searching information in the restaurant's wine list
    """
    question = state["question"]
    docs = search_wine.invoke(question)

    if len(docs) > 0:
        return {"documents": docs}
    else:
        return {"documents": [Document(page_content="관련 와인 정보를 찾을 수 없습니다.")]}


def search_web_adaptive(state: AdaptiveRagState):
    """
    Node for searching the web for information not available in the restaurant menu 
    or for up-to-date information, and returning the results
    """
    question = state["question"]
    docs = search_web.invoke(question)
    
    if len(docs) > 0:
        return {"documents": docs}
    else:
        return {"documents": [Document(page_content="관련 정보를 찾을 수 없습니다.")]}

`(4) 생성 노드`

In [17]:

# RAG 프롬프트 정의
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an assistant answering questions based on provided documents. Follow these guidelines:

1. Use only information from the given documents.
2. If the document lacks relevant info, say "The provided documents don't contain information to answer this question."
3. Cite relevant parts of the document in your answers.
4. Don't speculate or add information not in the documents.
5. Keep answers concise and clear.
6. Omit irrelevant information."""
),
    ("human", "Answer the following question using these documents:\n\n[Documents]\n{documents}\n\n[Question]\n{question}"),
])

def generate_adaptive(state: AdaptiveRagState):
    """
    Generate answer using the retrieved_documents
    """
    question = state.get("question", None)
    documents = state.get("documents", [])
    if not isinstance(documents, list):
        documents = [documents]

    # 문서 내용을 문자열로 변환
    documents_text = "\n\n".join([f"---\n본문: {doc.page_content}\n메타데이터:{str(doc.metadata)}\n---" for doc in documents])

    # RAG generation
    rag_chain = rag_prompt | llm | StrOutputParser()
    generation = rag_chain.invoke({"documents": documents_text, "question": question})
    return {"generation": generation}

In [18]:
# LLM Fallback 프롬프트 정의
fallback_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant helping with various topics. Follow these guidelines:

1. Provide accurate and helpful information to the best of your ability.
2. Express uncertainty when unsure; avoid speculation.
3. Keep answers concise yet informative.
4. Inform users they can ask for clarification if needed.
5. Respond ethically and constructively.
6. Mention reliable general sources when applicable."""),
    ("human", "{question}"),
])

def llm_fallback_adaptive(state: AdaptiveRagState):
    """
    Generate answer using the LLM without context
    """
    question = state.get("question", "")
    
    # LLM chain
    llm_chain = fallback_prompt | llm | StrOutputParser()
    
    generation = llm_chain.invoke({"question": question})
    return {"generation": generation}

`(5) 그래프 연결`

In [21]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

# 그래프 구성
builder = StateGraph(AdaptiveRagState)

# 노드 추가
builder.add_node("search_menu", search_menu_adaptive)
builder.add_node("search_wine", search_wine_adaptive)
builder.add_node("search_web", search_web_adaptive)
builder.add_node("generate", generate_adaptive)
builder.add_node("llm_fallback", llm_fallback_adaptive)

# 엣지 추가
builder.add_conditional_edges(
    START,
    route_question_adaptive
)

builder.add_edge("search_menu", "generate")
builder.add_edge("search_wine", "generate")
builder.add_edge("search_web", "generate")
builder.add_edge("generate", END)
builder.add_edge("llm_fallback", END)

# 그래프 컴파일 
adaptive_rag = builder.compile()

# 그래프 시각화
#display(Image(adaptive_rag.get_graph().draw_mermaid_png()))

In [22]:
mermaid_code = adaptive_rag.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

Mermaid Code:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	search_menu(search_menu)
	search_wine(search_wine)
	search_web(search_web)
	generate(generate)
	llm_fallback(llm_fallback)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> llm_fallback;
	__start__ -.-> search_menu;
	__start__ -.-> search_web;
	__start__ -.-> search_wine;
	search_menu --> generate;
	search_web --> generate;
	search_wine --> generate;
	generate --> __end__;
	llm_fallback --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



* https://mermaid.live/ 에서  mermain_code 로 직접 확인한다.

* [Graph이미지](https://mermaidchart.com/play?utm_source=mermaid_live_editor&utm_medium=share#pako:eNp9kt1ygjAQhV-F2d7gjFgMKhg63tRH6FVLx4mwEaYhMCFMfxzfvStVirXtFbv5djfnLNlDWmUIHHZG1LnzsI4TndjNprHC0Md9uqtXfXZ3W6-eR5xzWZjGHgsbFCbNNyXq1h3EowF7LTS6g_iC4db9DjuyQ41GWHTPQXeqVLmRQqmtSF_cYTL6kos668V2cS9ViS-lvQnHm3grZzgj_oUPzPyDSfR_lNzGP7bkeMTP1uLLVfzNaNA1PCcdOdmOfy7rCqa0kGaN0slQilZZRxZK8RvJpC_lWNFNXo7FLrd8OmEXDd1P78q9qhZpYd-5f1FwXPVp3FZuFzKFMT2rIgNOahocQ4mmFMcc9ol2nARsjiUmwCk8yUkg0Qfqq4V-rKoSuDUtdZqq3eXnpK0zsr0uBL3Zsh9uyCOa-6rVFjibdyOA7-ENeEhWwsBn82C5mAazJcF34NNZNIkWLFqyeRTMQhYdxvDR3elPwjBgIWPTRcB8PwijwyfsSxe-)

In [23]:
# 그래프 실행
inputs = {"question": "스테이크 메뉴의 가격은 얼마인가요?"}
for output in adaptive_rag.stream(inputs):
    for key, value in output.items():
        print(f"Node '{key}':")
        print(f"State '{value.keys()}':")
        print(f"Value '{value}':")
    print("\n---\n")

# 최종 답변
print(value["generation"])

Node 'search_menu':
State 'dict_keys(['documents'])':
Value '{'documents': [Document(id='ae8b7a7f-dfdf-4e07-83e6-00bd8d741f49', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 1, 'menu_name': '시그니처 스테이크'}, page_content='1. 시그니처 스테이크\n   • 가격: ₩35,000\n   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스\n   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.'), Document(id='9fc6ee99-c962-47fc-bbf8-f6aa8906a545', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 8, 'menu_name': '안심 스테이크 샐러드'}, page_content='8. 안심 스테이크 샐러드\n   • 가격: ₩26,000\n   • 주요 식재료: 소고기 안심, 루꼴라, 체리 토마토, 발사믹 글레이즈\n   • 설명: 부드러운 안심 스테이크를 얇게 슬라이스하여 신선한 루꼴라 위에 올린 메인 요리 샐러드입니다. 체리 토마토와 파마산 치즈 플레이크로 풍미를 더하고, 발사믹 글레이즈로 마무리하여 고기의 풍미를 한층 끌어올렸습니다.'), Document(id='554ad13b-2d56-43e7-a0e9-2a65409148e7', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 2, 'menu_name': '트러플 리조또'}, page_content='2

In [24]:
# 그래프 실행
inputs = {"question": "푸이 퓌세 2019의 주요 품종은 무엇인가요?"}
for output in adaptive_rag.stream(inputs):
    for key, value in output.items():
        print(f"Node '{key}':")
        print(f"State '{value.keys()}':")
        pprint(f"Value '{value}':")
        #pprint(f"Value '{value.page_content}':")
    print("\n---\n")

# 최종 답변
print(value["generation"])

Node 'search_wine':
State 'dict_keys(['documents'])':
("Value '{'documents': [Document(id='5f5e8b02-ea70-4dfd-8950-91cfc110db13', "
 "metadata={'source': '../data/restaurant_wine.txt', 'menu_number': 5, "
 "'menu_name': '푸이 퓌세 2019'}, page_content='5. 푸이 퓌세 2019\\n   • 가격: "
 '₩95,000\\n   • 주요 품종: 소비뇽 블랑\\n   • 설명: 프랑스 루아르 지역의 대표적인 화이트 와인입니다. 구스베리, '
 '레몬, 라임의 상큼한 과실향과 함께 미네랄, 허브 노트가 특징적입니다. 날카로운 산도와 깔끔한 피니시가 인상적이며, 신선한 굴이나 해산물 '
 "요리와 탁월한 페어링을 이룹니다.'), Document(id='572fc5d9-a967-4378-9e2a-50622067c6ba', "
 "metadata={'source': '../data/restaurant_wine.txt', 'menu_number': 2, "
 "'menu_name': '돔 페리뇽 2012'}, page_content='2. 돔 페리뇽 2012\\n   • 가격: "
 '₩380,000\\n   • 주요 품종: 샤르도네, 피노 누아\\n   • 설명: 프랑스 샴페인의 대명사로 알려진 프레스티지 큐베입니다. '
 '시트러스, 백도, 브리오쉬의 아로마가 조화롭게 어우러지며, 미네랄리티가 돋보입니다. 섬세하고 지속적인 버블과 크리미한 무스, 긴 여운이 '
 "특징입니다. 우아함과 복잡성이 완벽한 균형을 이룹니다.'), "
 "Document(id='b4793486-b265-4edb-8fbe-55b570e789c2', metadata={'source': "
 "'../data/restaurant_wine.txt', 'menu_number': 7, 'menu_name': '풀리니

### 3-2. 사람의 개입 (Human-in-the-Loop)

- Human-in-the-Loop (HITL)는 AI 시스템에 인간의 판단과 개입을 통합하는 접근 방식
- AI의 자동화된 처리와 인간의 전문성을 결합하여 더 정확하고 신뢰할 수 있는 결과를 도출하는 것을 목표


`(1) 체크포인트 설정`

In [28]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

`(2) Breakpoint 추가`


In [29]:
# 컴파일 - 'generate' 노드 전에 중단점 추가
adaptive_rag_hitl = builder.compile(checkpointer=memory, interrupt_before=["generate"])


In [30]:

# 그래프 출력
#display(Image(adaptive_rag_hitl.get_graph().draw_mermaid_png()))
mermaid_code = adaptive_rag_hitl.get_graph().draw_mermaid()
print("Mermaid Code:")
print(mermaid_code)

Mermaid Code:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	search_menu(search_menu)
	search_wine(search_wine)
	search_web(search_web)
	generate(generate<hr/><small><em>__interrupt = before</em></small>)
	llm_fallback(llm_fallback)
	__end__([<p>__end__</p>]):::last
	__start__ -.-> llm_fallback;
	__start__ -.-> search_menu;
	__start__ -.-> search_web;
	__start__ -.-> search_wine;
	search_menu --> generate;
	search_web --> generate;
	search_wine --> generate;
	generate --> __end__;
	llm_fallback --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



* https://mermaid.live/ 에서  mermain_code 로 직접 확인한다.

* [Graph이미지](https://mermaidchart.com/play?utm_source=mermaid_live_editor&utm_medium=share#pako:eNp9Ul2PmzAQ_CuW-0IkSIhJAjGUl-Yn3NMdVWRgHVCNQcaovUb577chH0fu2nvyjGZ3vTPaIy3aEiinByO6ijzt4kxndr_vrTD4OC9Jl95ZsujSnzPOuaxNb8-FPQhTVPsG9OBM8Gyi_a41OBP8oEHuvMNROYAGIyw4N5BUZpEmfSOUShNocJtaWzBm6Cz5TnKQrYFkgUKyuBSNY5Rq9hJZLopfzpTMLv5Al3d3I757U-Ji7e6aeHMvJdMZ8T_0ifsvZHT5lYrxxB9iJR7qtyzix-z-r-Ggz-KNjMrVdvwxrE9igYH0O5CkBCkGZYmsleLfJJO-lK7Cn7wK6kNl-XLOHhrGKxnLvbYTRW1fuf9QcI76Oi6X-UYW1MU7rEvKcZseXNqAacSZ02OmCcmoraCBjHKE13UymukT9nVCP7dtQ7k1A3aadjhUNzJ0Jdre1QKP_L0CLYL50Q7aUs624wTKj_QP5SE6CQOfrYPtZhmstmuXvlK-XEXzaMOiLVtHwSpk0cmlf8cv_XkYBixkbLkJmO8HYXR6A5WEKLE)

`(3) Breakpoint 실행 확인`


In [31]:
# 도구 사용 전 중단점에서 실행을 멈춤 

thread = {"configurable": {"thread_id": "breakpoint_test"}}
inputs = {"question": "스테이크 메뉴의 가격은 얼마인가요?"}
for event in adaptive_rag_hitl.stream(inputs, config=thread):
    for k, v in event.items():
        # '__end__' 이벤트는 미출력
        if k != "__end__":
            print(f"{k}: {v}")  # 이벤트의 키와 값을 함께 출력

search_menu: {'documents': [Document(id='ae8b7a7f-dfdf-4e07-83e6-00bd8d741f49', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 1, 'menu_name': '시그니처 스테이크'}, page_content='1. 시그니처 스테이크\n   • 가격: ₩35,000\n   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스\n   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.'), Document(id='9fc6ee99-c962-47fc-bbf8-f6aa8906a545', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 8, 'menu_name': '안심 스테이크 샐러드'}, page_content='8. 안심 스테이크 샐러드\n   • 가격: ₩26,000\n   • 주요 식재료: 소고기 안심, 루꼴라, 체리 토마토, 발사믹 글레이즈\n   • 설명: 부드러운 안심 스테이크를 얇게 슬라이스하여 신선한 루꼴라 위에 올린 메인 요리 샐러드입니다. 체리 토마토와 파마산 치즈 플레이크로 풍미를 더하고, 발사믹 글레이즈로 마무리하여 고기의 풍미를 한층 끌어올렸습니다.'), Document(id='554ad13b-2d56-43e7-a0e9-2a65409148e7', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 2, 'menu_name': '트러플 리조또'}, page_content='2. 트러플 리조또\n   • 가격: ₩22,000\n   • 주요 식재료: 이탈리아산 

`(4) Breakpoint 상태 관리`


In [32]:
# 상태 확인
current_state = adaptive_rag_hitl.get_state(thread)
print("---그래프 상태---")
print(current_state)
print("-"*50)
print(current_state.values.get("generation"))

---그래프 상태---
StateSnapshot(values={'question': '스테이크 메뉴의 가격은 얼마인가요?', 'documents': [Document(id='ae8b7a7f-dfdf-4e07-83e6-00bd8d741f49', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 1, 'menu_name': '시그니처 스테이크'}, page_content='1. 시그니처 스테이크\n   • 가격: ₩35,000\n   • 주요 식재료: 최상급 한우 등심, 로즈메리 감자, 그릴드 아스파라거스\n   • 설명: 셰프의 특제 시그니처 메뉴로, 21일간 건조 숙성한 최상급 한우 등심을 사용합니다. 미디엄 레어로 조리하여 육즙을 최대한 보존하며, 로즈메리 향의 감자와 아삭한 그릴드 아스파라거스가 곁들여집니다. 레드와인 소스와 함께 제공되어 풍부한 맛을 더합니다.'), Document(id='9fc6ee99-c962-47fc-bbf8-f6aa8906a545', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 8, 'menu_name': '안심 스테이크 샐러드'}, page_content='8. 안심 스테이크 샐러드\n   • 가격: ₩26,000\n   • 주요 식재료: 소고기 안심, 루꼴라, 체리 토마토, 발사믹 글레이즈\n   • 설명: 부드러운 안심 스테이크를 얇게 슬라이스하여 신선한 루꼴라 위에 올린 메인 요리 샐러드입니다. 체리 토마토와 파마산 치즈 플레이크로 풍미를 더하고, 발사믹 글레이즈로 마무리하여 고기의 풍미를 한층 끌어올렸습니다.'), Document(id='554ad13b-2d56-43e7-a0e9-2a65409148e7', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 2, 'menu_name': '트러플 리조또'}, page_co

In [33]:
# 다음에 실행될 노드를 확인 
current_state.next

('generate',)

`(5) Breakpoint 이후 단계를 계속해서 실행`

In [34]:
# 입력값을 None으로 지정하면 중단점부터 실행하는 의미 
for event in adaptive_rag_hitl.stream(None, config=thread):
    for k, v in event.items():
        # '__end__' 이벤트는 미출력
        if k != "__end__":
            print(f"{k}: {v}")  # 이벤트의 키와 값을 함께 출력

generate: {'generation': '스테이크 메뉴의 가격은 다음과 같습니다:  \n1. **시그니처 스테이크**: ₩35,000 (문서 1번 메뉴)  \n2. **안심 스테이크 샐러드**: ₩26,000 (문서 8번 메뉴)  \n\n**인용된 문서 내용**:  \n- "가격: ₩35,000" (시그니처 스테이크)  \n- "가격: ₩26,000" (안심 스테이크 샐러드)'}


In [35]:
# 다음에 실행될 노드를 확인 
current_state = adaptive_rag_hitl.get_state(thread)
current_state.next

()

In [36]:
# 최종 답변
current_state = adaptive_rag_hitl.get_state(thread)
print(current_state.values.get("generation"))

스테이크 메뉴의 가격은 다음과 같습니다:  
1. **시그니처 스테이크**: ₩35,000 (문서 1번 메뉴)  
2. **안심 스테이크 샐러드**: ₩26,000 (문서 8번 메뉴)  

**인용된 문서 내용**:  
- "가격: ₩35,000" (시그니처 스테이크)  
- "가격: ₩26,000" (안심 스테이크 샐러드)


`(6) 상태 업데이트`

In [37]:
# 새로운 thread를 생성하고, 새로운 질문을 수행 
thread = {"configurable": {"thread_id": "breakpoint_update"}}
inputs = {"question": "매운 음식이 있나요?"}
for event in adaptive_rag_hitl.stream(inputs, config=thread):
    for k, v in event.items():
        if k != "__end__":
            print(f"{k}: {v}") 

search_menu: {'documents': [Document(id='ec4f3da2-2b8d-48c3-90fb-9d5213ff4d13', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 10, 'menu_name': '티라미수'}, page_content='10. 티라미수\n    • 가격: ₩9,000\n    • 주요 식재료: 마스카포네 치즈, 에스프레소, 카카오 파우더, 레이디핑거 비스킷\n    • 설명: 부드러운 마스카포네 치즈 크림과 에스프레소에 적신 레이디핑거 비스킷을 층층이 쌓아 만든 이탈리아 정통 디저트입니다. 고소한 카카오 파우더를 듬뿍 뿌려 풍미를 더했습니다. 커피의 쌉싸름함과 치즈의 부드러움이 조화롭게 어우러집니다.'), Document(id='3d4eafa5-7417-4bbf-98ee-7ca875e52336', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 9, 'menu_name': '치킨 콘피'}, page_content='9. 치킨 콘피\n   • 가격: ₩23,000\n   • 주요 식재료: 닭다리살, 허브, 마늘, 올리브 오일\n   • 설명: 닭다리살을 허브와 마늘을 넣은 올리브 오일에 저온에서 장시간 조리한 프랑스 요리입니다. 부드럽고 촉촉한 육질이 특징이며, 로즈메리 감자와 제철 채소를 곁들여 제공합니다. 레몬 제스트를 뿌려 상큼한 향을 더했습니다.'), Document(id='aa4d42e1-47a6-475b-9d09-5d6ca87996dd', metadata={'source': '../data/restaurant_menu.txt', 'menu_number': 4, 'menu_name': '버섯 크림 수프'}, page_content='4. 버섯 크림 수프\n   • 가격: ₩10,000\n   • 주요 식재료: 양송이버섯, 표고버섯, 생크림, 트러플 오일\n   • 설명: 양

In [38]:
# 다음에 실행될 노드를 확인 
current_state = adaptive_rag_hitl.get_state(thread)
current_state.next

('generate',)

In [39]:
# question, generation 필드 확인
current_state = adaptive_rag_hitl.get_state(thread)
print(current_state.values.get("question"))
print("-"*50)
print(current_state.values.get("generation"))

매운 음식이 있나요?
--------------------------------------------------
None


In [40]:
# 상태 업데이트 - 질문을 수정하여 업데이트
adaptive_rag_hitl.update_state(thread, {"question": "매콤한 해산물 요리가 있나요?"})

# 상태 확인
new_state = adaptive_rag_hitl.get_state(thread)

print(new_state.values.get("question"))
print("-"*50)
print(new_state.values.get("generation"))

매콤한 해산물 요리가 있나요?
--------------------------------------------------
None


In [41]:
# 입력값을 None으로 지정하면 중단점부터 실행하고 최종 답변을 생성 
for event in adaptive_rag_hitl.stream(None, config=thread):
    for k, v in event.items():
        # '__end__' 이벤트는 미출력
        if k != "__end__":
            print(f"{k}: {v}")  # 이벤트의 키와 값을 함께 출력

generate: {'generation': 'The provided documents don\'t contain information to answer this question. \n\nNone of the listed menu items are described as "매콤한" (spicy) or explicitly feature seafood prominently beyond the lobster bisque, which is not described as spicy. The closest seafood option is "랍스터 비스크" (lobster bisque), but its description emphasizes richness from cream and brandy rather than spiciness.'}


In [42]:
# 최종 답변 확인
print(event["generate"]["generation"])

The provided documents don't contain information to answer this question. 

None of the listed menu items are described as "매콤한" (spicy) or explicitly feature seafood prominently beyond the lobster bisque, which is not described as spicy. The closest seafood option is "랍스터 비스크" (lobster bisque), but its description emphasizes richness from cream and brandy rather than spiciness.
